# Gmail + Gemini: limpieza inteligente en Colab

Este notebook evita el flujo bloqueado de `google.colab.auth` para Gmail.

Flujo correcto:

1. Generas `token.json` una sola vez en tu PC local con `credentials.json`.
2. Subes `credentials.json` y `token.json` a Colab.
3. El notebook usa esos archivos para Gmail y `GEMINI_API_KEY` desde Colab Secrets.

Fecha de referencia: 9 de julio de 2026. El modelo por defecto es `gemini-3.5-flash`, reemplazo recomendado actual de `gemini-2.5-flash`.

## Antes de empezar

Necesitas:

- `credentials.json` creado en Google Cloud como OAuth Client de tipo `Desktop app`
- `token.json` generado localmente con ese mismo `credentials.json`
- `GEMINI_API_KEY` guardada en Colab Secrets

Si todavia no tienes `token.json`, generarlo con el script local que te dejo junto a este notebook.

In [ ]:
!pip -q install -U google-genai google-api-python-client google-auth-httplib2 google-auth-oauthlib pydantic beautifulsoup4

## Paso 1: subir `credentials.json` y `token.json`

Ejecuta la celda y sube ambos archivos.

In [ ]:
from google.colab import files

uploaded = files.upload()

required_files = {"credentials.json", "token.json"}
missing = required_files.difference(uploaded.keys())
if missing:
    raise ValueError(
        "Faltan archivos requeridos: " + ", ".join(sorted(missing)) + ". "
        "Genera token.json en tu PC local y vuelvelos a subir."
    )

print("Archivos subidos correctamente")

In [ ]:
from __future__ import annotations

import base64
import json
import os
import re
from email.utils import parsedate_to_datetime
from typing import Literal, Optional

from bs4 import BeautifulSoup
from google import genai
from google.auth.transport.requests import Request
from google.colab import userdata
from google.genai import types
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from pydantic import BaseModel, Field

SCOPES = ["https://www.googleapis.com/auth/gmail.modify"]

dry_run = True
MAX_EMAILS = 30
MODEL_NAME = "gemini-3.5-flash"

# Cambia estas variables si luego quieres otra politica de borrado.
DELETE_ONLY_IF_CATEGORY = "Promociones Irrelevantes"
DELETE_MIN_CONFIDENCE = 75


def get_gmail_credentials() -> Credentials:
    if not os.path.exists("token.json"):
        raise FileNotFoundError(
            "No existe token.json. Generarlo primero en tu PC local y subirlo a Colab."
        )

    creds = Credentials.from_authorized_user_file("token.json", SCOPES)

    if creds.expired and creds.refresh_token:
        creds.refresh(Request())
        with open("token.json", "w", encoding="utf-8") as token_file:
            token_file.write(creds.to_json())

    if not creds.valid:
        raise RuntimeError(
            "token.json no es valido para estos scopes. Regeneralo localmente con gmail.modify."
        )

    return creds


def build_gmail_service():
    creds = get_gmail_credentials()
    service = build("gmail", "v1", credentials=creds, cache_discovery=False)
    service.users().getProfile(userId="me").execute()
    return service


gmail_service = build_gmail_service()

api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise ValueError(
        "No se encontro el secreto GEMINI_API_KEY en Colab Secrets. Guardalo y vuelve a correr."
    )

gemini_client = genai.Client(api_key=api_key)

print(f"dry_run = {dry_run}")
print(f"Modelo Gemini = {MODEL_NAME}")
print("Gmail autenticado correctamente")

In [ ]:
def get_header_value(headers: list[dict], name: str, default: str = "") -> str:
    for header in headers or []:
        if header.get("name", "").lower() == name.lower():
            return header.get("value", default)
    return default


def decode_base64url(data: str) -> str:
    if not data:
        return ""
    padding = "=" * (-len(data) % 4)
    raw = base64.urlsafe_b64decode(data + padding)
    return raw.decode("utf-8", errors="replace")


def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def html_to_text(html: str) -> str:
    if not html:
        return ""
    return clean_text(BeautifulSoup(html, "html.parser").get_text(" "))


def extract_message_text(payload: Optional[dict]) -> str:
    if not payload:
        return ""

    mime_type = payload.get("mimeType", "")
    body_data = payload.get("body", {}).get("data")

    if body_data:
        decoded = decode_base64url(body_data)
        if mime_type == "text/html":
            return html_to_text(decoded)
        return clean_text(decoded)

    for part in payload.get("parts", []) or []:
        text = extract_message_text(part)
        if text:
            return text

    return ""


def normalize_date(date_value: str) -> str:
    if not date_value:
        return ""
    try:
        return parsedate_to_datetime(date_value).isoformat()
    except Exception:
        return date_value


def fetch_recent_inbox_messages(service, max_results: int = 30) -> list[dict]:
    response = service.users().messages().list(
        userId="me",
        labelIds=["INBOX"],
        maxResults=max_results,
    ).execute()

    detailed_messages = []
    for item in response.get("messages", []):
        message_id = item.get("id")
        if not message_id:
            continue

        try:
            msg = service.users().messages().get(
                userId="me",
                id=message_id,
                format="full",
            ).execute()

            payload = msg.get("payload", {})
            headers = payload.get("headers", [])
            body_text = extract_message_text(payload)
            snippet = clean_text(msg.get("snippet", ""))
            preview = (body_text or snippet)[:500]

            detailed_messages.append(
                {
                    "id": msg.get("id", ""),
                    "date": normalize_date(get_header_value(headers, "Date")),
                    "from": get_header_value(headers, "From", "(Sin remitente)"),
                    "subject": get_header_value(headers, "Subject", "(Sin asunto)"),
                    "snippet": preview,
                }
            )
        except Exception as exc:
            print(f"[WARN] No se pudo leer el mail {message_id}: {exc}")

    return detailed_messages


emails = fetch_recent_inbox_messages(gmail_service, max_results=MAX_EMAILS)
print(f"Correos listos para clasificar: {len(emails)}")
emails[:3]

In [ ]:
class EmailClassification(BaseModel):
    categoria: Literal[
        "Importante",
        "Facturas y Comprobantes",
        "Promociones Irrelevantes",
        "Notificaciones de Sistemas",
    ]
    confianza_porcentaje: int = Field(ge=0, le=100)
    eliminar: bool


CLASSIFICATION_PROMPT = """
Sos un clasificador estricto de correos de Gmail.

Debes devolver EXCLUSIVAMENTE un JSON valido con este schema:
{
  "categoria": "Importante" | "Facturas y Comprobantes" | "Promociones Irrelevantes" | "Notificaciones de Sistemas",
  "confianza_porcentaje": numero entero entre 0 y 100,
  "eliminar": true | false
}

Reglas:
- Importante: correos personales, laborales o utiles que requieren seguimiento.
- Facturas y Comprobantes: facturas, tickets, recibos, pagos y comprobantes.
- Promociones Irrelevantes: publicidad agresiva, ofertas poco utiles, newsletters comerciales descartables.
- Notificaciones de Sistemas: accesos, resets, alertas automaticas, monitoreo y mensajes de plataformas.

Regla estricta de eliminacion:
- `eliminar` debe ser true SOLO si la categoria es `Promociones Irrelevantes` y la confianza_porcentaje es mayor a 75.
- En cualquier otro caso, `eliminar` debe ser false.

No agregues texto adicional.
""".strip()


def classify_email_with_gemini(client, email_data: dict) -> EmailClassification:
    payload = {
        "id": email_data["id"],
        "date": email_data["date"],
        "from": email_data["from"],
        "subject": email_data["subject"],
        "snippet": email_data["snippet"],
    }

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=[
            {"role": "user", "parts": [{"text": CLASSIFICATION_PROMPT}]},
            {
                "role": "user",
                "parts": [
                    {
                        "text": "Clasifica este correo y aplica la regla estricta de eliminacion:\n" + json.dumps(payload, ensure_ascii=False)
                    }
                ],
            },
        ],
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",
            response_schema=EmailClassification,
        ),
    )

    if response.parsed:
        return response.parsed

    raw_text = getattr(response, "text", "") or ""
    return EmailClassification.model_validate_json(raw_text)


In [ ]:
results = []
processed = 0
simulated_trash = 0
real_trash = 0
errors = 0

for email_data in emails:
    processed += 1
    subject = email_data.get("subject", "(Sin asunto)")

    try:
        decision = classify_email_with_gemini(gemini_client, email_data)

        should_delete = (
            decision.categoria == DELETE_ONLY_IF_CATEGORY
            and decision.confianza_porcentaje > DELETE_MIN_CONFIDENCE
        )

        action = "MANTENER"
        if should_delete and dry_run:
            action = "MOVER A PAPELERA (Simulado)"
            simulated_trash += 1
        elif should_delete and not dry_run:
            gmail_service.users().messages().trash(
                userId="me",
                id=email_data["id"],
            ).execute()
            action = "MOVER A PAPELERA"
            real_trash += 1

        print(f"[{subject}] -> {decision.categoria} ({decision.confianza_porcentaje}%) -> Accion: {action}")

        results.append(
            {
                **email_data,
                **decision.model_dump(),
                "accion": action,
            }
        )

    except HttpError as exc:
        errors += 1
        print(f"[ERROR][Gmail] {subject}: {exc}")
    except Exception as exc:
        errors += 1
        print(f"[ERROR][Gemini/Parseo] {subject}: {exc}")

print("\nResumen")
print(f"- Procesados: {processed}")
print(f"- Simulados a papelera: {simulated_trash}")
print(f"- Enviados realmente a papelera: {real_trash}")
print(f"- Errores: {errors}")